# `TRG_DEP.txt` Conversion**Conversion Timestamp:** 2024-07-30 12:00:00 UTC**Description:** This notebook converts the `TRG_DEP.txt` ODI SQL script into Databricks Spark SQL. It performs a direct insert from the `departments` table into the `trg_dep` target table.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")dbutils.widgets.text("DATASOURCE_NUM_ID", "-1", "Datasource Number ID")dbutils.widgets.text("ETL_PROC_WID", "-1", "ETL Process Widget ID")dbutils.widgets.text("ODI_SESS_NO", "-1", "ODI Session Number")

# ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time ASSELECT current_timestamp() AS etl_current_extract_time;

In [ ]:
current_extract_time = spark.sql("SELECT etl_current_extract_time FROM v_etl_current_extract_time").collect()[0].etl_current_extract_timeprint(f"ETL_JOB_TYPE: {dbutils.widgets.get('ETL_JOB_TYPE')}")print(f"DATASOURCE_NUM_ID: {dbutils.widgets.get('DATASOURCE_NUM_ID')}")print(f"ETL_PROC_WID: {dbutils.widgets.get('ETL_PROC_WID')}")print(f"ODI_SESS_NO: {dbutils.widgets.get('ODI_SESS_NO')}")print(f"ETL Current Extract Time: {current_extract_time}")

# Staging Table

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Initial step (no operation)-- SCEN_TASK_NO in {20}: Initial step (no operation)-- SCEN_TASK_NO in {30}: Initial step (no operation)INSERT INTO workspace.hr.trg_dep  (    department_id ,    department_name ,    manager_id ,    location_id  )SELECT  departments.department_id ,  departments.department_name ,  departments.manager_id ,  departments.location_idFROM  workspace.hr.departments AS departments;

In [ ]:
%sql
SELECT COUNT(*) AS inserted_rows FROM workspace.hr.trg_dep;

# Cleanup

# Validation

In [ ]:
%sql
SELECT * FROM workspace.hr.trg_dep LIMIT 100;

# Conversion Notes and Manual Actions Required1.  **Schema and Table Names:** All schema names have been converted to `workspace.hr` and table names to lowercase (`trg_dep`, `departments`) as per the naming conventions.2.  **Oracle Hints:** The Oracle `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Databricks Delta Lake.3.  **Data Types:** The `INSERT` statement does not explicitly define column types. It is assumed the `workspace.hr.trg_dep` table is pre-existing with compatible Spark SQL data types (e.g., `BIGINT` for `DEPARTMENT_ID`, `STRING` for `DEPARTMENT_NAME`). If `TRG_DEP` needs to be created, ensure the DDL uses appropriate Spark data types (e.g., `NUMBER(p,0)` -> `BIGINT`, `VARCHAR2` -> `STRING`).4.  **SCEN_TASK_NO Handling:** The empty `SCEN_TASK_NO {10}, {20}, {30}` steps from the original ODI file have been included as comments in the relevant SQL cell, as they contained no executable SQL.